<a href="https://colab.research.google.com/github/binary-cpu/flyrank-ml-track-david/blob/main/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question.** Which content pages should an editor look at first? Using only March 2026 search data, can we rank pages so that those losing at least 20% of impressions in April appear near the top, and does a model do that better than a simple rule?

**Decision supported:** the order in which a human reviews a refresh backlog. **Not** a forecast, and not a claim that refreshing helps.

In [1]:
%pip -q install duckdb scikit-learn matplotlib
import os, json, io, base64, zipfile, getpass, warnings
import numpy as np, pandas as pd, duckdb
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

FEATURE_MONTH, OUTCOME_MONTH, SEED = "2026-03", "2026-04", 42
try:                                   # Colab Secret first (never paste a token into a cell)
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your HF READ token: ")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"

df = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
      SUM(CASE WHEN month='{FEATURE_MONTH}' THEN gsc_impressions ELSE 0 END) AS imp_march,
      SUM(CASE WHEN month='{FEATURE_MONTH}' THEN gsc_clicks ELSE 0 END) AS clk_march,
      AVG(CASE WHEN month='{FEATURE_MONTH}' AND gsc_avg_position>0 THEN gsc_avg_position END) AS pos_march,
      COUNT(*) FILTER (WHERE month='{FEATURE_MONTH}' AND gsc_impressions>0) AS active_days,
      SUM(CASE WHEN month='{OUTCOME_MONTH}' THEN gsc_impressions ELSE 0 END) AS imp_april
    FROM {FACT} WHERE month IN ('{FEATURE_MONTH}','{OUTCOME_MONTH}')
    GROUP BY 1,2 HAVING imp_march >= 100
    ORDER BY 1,2""").df()          # ORDER BY pins row order, so reruns reproduce exactly
dim = con.sql(f"""SELECT content_hash_id,
    DATE_DIFF('day', content_updated_date, DATE '{FEATURE_MONTH}-01') AS days FROM {DIM}""").df()
model_df = df.merge(dim, on="content_hash_id", how="left")
model_df.loc[model_df["days"] < 0, "days"] = np.nan     # updated AFTER March: unknowable as of March
stats = dict(
    n_daily=int(con.sql(f"SELECT COUNT(*) FROM {FACT} WHERE month IN ('{FEATURE_MONTH}','{OUTCOME_MONTH}')").fetchone()[0]),
    ga4_pct=float(con.sql(f"""SELECT 100.0*COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)/COUNT(*)
                              FROM {FACT} WHERE month='{FEATURE_MONTH}'""").fetchone()[0]))
print(f"{len(model_df):,} content items, {model_df.client_hash_id.nunique()} clients")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,441 content items, 44 clients


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Data.** `fact_content_daily_performance` (daily search metrics) joined to `dim_content` (update date). Feature month: March 2026. Outcome month: April 2026. Final month sealed. Kept items with 100+ March impressions and a measured position. Excluded product flags, identifiers as inputs, and GA4 fields (sparse).

In [2]:
model_df = model_df.dropna(subset=["pos_march"]).reset_index(drop=True)
model_df["ctr"] = model_df.clk_march / model_df.imp_march
model_df["declined"] = (model_df.imp_april < 0.8 * model_df.imp_march).astype(int)   # LABEL: >=20% impression drop
model_df["stale_unknown"] = model_df.days.isna().astype(int)
model_df["days_f"] = model_df.days.fillna(-1)
FEATURES = ["imp_march", "ctr", "pos_march", "active_days", "days_f", "stale_unknown"]  # March only
X, y, g = model_df[FEATURES], model_df.declined, model_df.client_hash_id
stats.update(n_items=len(model_df), n_clients=int(g.nunique()), base_rate=float(y.mean()))
print(stats)

{'n_daily': 20266108, 'ga4_pct': 4.206382480177065, 'n_items': 101441, 'n_clients': 44, 'base_rate': 0.5174633530820871}


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Method.** Label = April impressions below 80% of March. Features are March-only. Baselines: the week-4 stale+visible rule and a lowest-CTR rule. Models: Logistic Regression and Random Forest. Primary split is grouped by client; five grouped folds check stability. Leakage audit compares random vs grouped splits and adds a deliberate leak to prove the harness reacts.

In [3]:
def pk(yv, s, k):
    return float(np.asarray(yv)[np.argsort(-np.asarray(s), kind="stable")[:k]].mean())
def score_all(yv, s):
    return dict(auc=float(roc_auc_score(yv, s)), p50=pk(yv, s, 50), p500=pk(yv, s, 500))
def make_rf(): return RandomForestClassifier(n_estimators=200, max_depth=6, random_state=SEED, n_jobs=-1)
def make_lr(): return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED))
def rule_stale(d): return ((d.days >= 180).fillna(False) & (d.imp_march >= 500)).astype(int) * d.imp_march  # week-4 baseline
def rule_ctr(d): return -d.ctr                                                                            # simplest rule: lowest CTR first

tr, te = next(GroupShuffleSplit(1, test_size=0.25, random_state=SEED).split(X, y, g))   # PRIMARY: grouped by client
Xtr, Xte, ytr, yte, dte = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te], model_df.iloc[te]
rf, lr = make_rf().fit(Xtr, ytr), make_lr().fit(Xtr, ytr)
res = {"Week-4 rule (stale + visible)": score_all(yte, rule_stale(dte)),
       "Simple rule (lowest CTR first)": score_all(yte, rule_ctr(dte)),
       "Logistic Regression": score_all(yte, lr.predict_proba(Xte)[:, 1]),
       "Random Forest": score_all(yte, rf.predict_proba(Xte)[:, 1])}
split = dict(test_clients=int(dte.client_hash_id.nunique()), test_rows=len(te), test_base=float(yte.mean()))

# leakage audit: random vs grouped split, and a deliberate leak to prove the harness reacts
from sklearn.model_selection import train_test_split
rtr, rte = train_test_split(np.arange(len(X)), test_size=0.25, random_state=SEED, stratify=y)
auc_random = roc_auc_score(y.iloc[rte], make_rf().fit(X.iloc[rtr], y.iloc[rtr]).predict_proba(X.iloc[rte])[:, 1])
XL = X.assign(imp_april_LEAK=model_df.imp_april)
auc_leak = roc_auc_score(y.iloc[rte], make_rf().fit(XL.iloc[rtr], y.iloc[rtr]).predict_proba(XL.iloc[rte])[:, 1])
audit = dict(random=float(auc_random), grouped=res["Random Forest"]["auc"], leak=float(auc_leak))
pd.DataFrame(res).T.round(3).assign(base_rate=round(split["test_base"], 3)), audit

(                                  auc   p50   p500  base_rate
 Week-4 rule (stale + visible)   0.500  0.28  0.320      0.484
 Simple rule (lowest CTR first)  0.612  0.42  0.404      0.484
 Logistic Regression             0.648  0.38  0.624      0.484
 Random Forest                   0.673  0.64  0.610      0.484,
 {'random': 0.6752332018636646,
  'grouped': 0.6725788297975568,
  'leak': 0.9171616211664817})

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**Results** use the same grouped test split for every method, with the base rate beside every precision number. Out-of-fold scores (a model that never saw the client) feed the queue below.

In [4]:
# stability: 5 client-grouped folds (each client is test exactly once); out-of-fold scores feed the queue
folds = {k: [] for k in ["Week-4 rule (stale + visible)", "Simple rule (lowest CTR first)", "Logistic Regression", "Random Forest"]}
model_df["oof_score"] = 0.0
for a, b in GroupKFold(5).split(X, y, g):
    d = model_df.iloc[b]
    p = make_rf().fit(X.iloc[a], y.iloc[a]).predict_proba(X.iloc[b])[:, 1]
    model_df.loc[model_df.index[b], "oof_score"] = p
    folds["Random Forest"].append(roc_auc_score(y.iloc[b], p))
    folds["Logistic Regression"].append(roc_auc_score(y.iloc[b], make_lr().fit(X.iloc[a], y.iloc[a]).predict_proba(X.iloc[b])[:, 1]))
    folds["Week-4 rule (stale + visible)"].append(roc_auc_score(y.iloc[b], rule_stale(d)))
    folds["Simple rule (lowest CTR first)"].append(roc_auc_score(y.iloc[b], rule_ctr(d)))
cv = {k: (float(np.mean(v)), float(np.std(v))) for k, v in folds.items()}
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
pd.DataFrame(cv, index=["mean AUC", "sd"]).T.round(3), imp.round(3)

(                                mean AUC     sd
 Week-4 rule (stale + visible)      0.500  0.000
 Simple rule (lowest CTR first)     0.599  0.010
 Logistic Regression                0.646  0.011
 Random Forest                      0.642  0.018,
 ctr              0.377
 active_days      0.330
 imp_march        0.126
 pos_march        0.090
 days_f           0.043
 stale_unknown    0.034
 dtype: float64)

## 5. Limitations

*What this work cannot claim.*

**Limitations** are computed here, not asserted: unknown update dates, the repeated update-date value, and zero-click items all come from the data. See the paper's Limitations section for the framing.

In [5]:
known = model_df.days.dropna()
lim = dict(unk_stale=100 * float(model_df.days.isna().mean()),
           zero_ctr=100 * float((model_df.ctr == 0).mean()),
           modal_days=int(known.mode().iloc[0]), modal_pct=100 * float((known == known.mode().iloc[0]).mean()),
           pos_lt1=100 * float((model_df.pos_march < 1).mean()))
lim

{'unk_stale': 82.24090850839404,
 'zero_ctr': 37.242337910706716,
 'modal_days': 4,
 'modal_pct': 96.16430752150985,
 'pos_lt1': 0.28883784663006085}

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

**Recommendations.** The top 10% by out-of-fold score become 'high risk' and get an action from the strongest visible reason. Cost/value is measured as extra declining pages surfaced per 500 reviews versus random review.

In [6]:
thr = model_df.oof_score.quantile(0.90)
hi = model_df.oof_score >= thr
stale = model_df.days.notna() & (model_df.days >= 180)
lowctr = (model_df.ctr < 0.005) & (model_df.pos_march <= 20)
model_df["reason_code"] = np.select([hi & stale, hi & lowctr, hi],
                                    ["stale_high_risk", "low_ctr_high_risk", "high_risk_unclear_cause"], "stable")
ACTION = {"stale_high_risk": "Review for content refresh", "low_ctr_high_risk": "Review title/snippet and CTA",
          "high_risk_unclear_cause": "Manual review (cause unclear)", "stable": "Monitor"}
model_df["action"] = model_df.reason_code.map(ACTION)
queue = model_df.sort_values("oof_score", ascending=False).reset_index(drop=True)
counts = queue.reason_code.value_counts().to_dict()
top500 = queue.head(500)
value = dict(p500=float(top500.declined.mean()), base=stats["base_rate"])
value["extra"] = (value["p500"] - value["base"]) * 500          # extra declining pages surfaced vs random review
value["wasted"] = (1 - value["p500"]) * 500
value["top_zero"] = 100 * float((top500.ctr == 0).mean())                     # reviewed pages that did NOT decline
top10 = queue.head(10)[["reason_code", "action", "imp_march", "ctr", "pos_march", "oof_score"]].round(3)
counts, value, top10

({'stable': 91296,
  'low_ctr_high_risk': 8562,
  'high_risk_unclear_cause': 1582,
  'stale_high_risk': 1},
 {'p500': 0.616,
  'base': 0.5174633530820871,
  'extra': 49.26832345895643,
  'wasted': 192.0,
  'top_zero': 59.8},
                reason_code                         action  imp_march    ctr  \
 0  high_risk_unclear_cause  Manual review (cause unclear)      650.0  0.000   
 1        low_ctr_high_risk   Review title/snippet and CTA      609.0  0.000   
 2        low_ctr_high_risk   Review title/snippet and CTA     2191.0  0.000   
 3        low_ctr_high_risk   Review title/snippet and CTA     3456.0  0.001   
 4        low_ctr_high_risk   Review title/snippet and CTA     1294.0  0.001   
 5  high_risk_unclear_cause  Manual review (cause unclear)      263.0  0.000   
 6        low_ctr_high_risk   Review title/snippet and CTA     1746.0  0.001   
 7        low_ctr_high_risk   Review title/snippet and CTA     1509.0  0.000   
 8        low_ctr_high_risk   Review title/snippet and 

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

**Artifacts.** This cell writes two charts, a metrics JSON, the queue CSV (stays out of git), the paper page `docs/index.html`, and `paper_bundle.zip` to download. Numbers in the page come from the same variables as the tables above, so they always match.

In [7]:
os.makedirs("docs", exist_ok=True); os.makedirs("work/figures", exist_ok=True); os.makedirs("work/outputs", exist_ok=True)
names = list(res); short = ["Week-4\nrule", "Lowest-CTR\nrule", "Logistic\nReg.", "Random\nForest"]
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].bar(short, [res[n]["auc"] for n in names], color="#1d4ed8"); ax[0].axhline(0.5, ls="--", c="grey"); ax[0].set_ylim(0.4, 0.8)
ax[0].set_title("AUC (0.5 = chance)")
ax[1].bar(short, [res[n]["p500"] for n in names], color="#1d4ed8"); ax[1].axhline(split["test_base"], ls="--", c="grey")
ax[1].set_ylim(0, 1); ax[1].set_title("Precision@500 (dashed = base rate)")
plt.tight_layout(); plt.savefig("work/figures/capstone_model_vs_baseline.png", dpi=150); plt.close()
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(["Random split", "Grouped by client", "Deliberate leak\n(April impressions added)"],
       [audit["random"], audit["grouped"], audit["leak"]], color=["#6b7280", "#1d4ed8", "#b91c1c"]); ax.set_ylim(0.4, 1.0)
ax.set_title("Random Forest AUC under three designs"); plt.tight_layout()
plt.savefig("work/figures/capstone_leakage_audit.png", dpi=150); plt.close()
b64 = lambda p: base64.b64encode(open(p, "rb").read()).decode()

f = lambda x, n=3: f"{x:.{n}f}"
rfm, ctrm = res["Random Forest"], res["Simple rule (lowest CTR first)"]
lrm, wk = res["Logistic Regression"], res["Week-4 rule (stale + visible)"]
best_simple = max(ctrm["auc"], wk["auc"])
if abs(rfm["auc"] - lrm["auc"]) <= 0.02:
    tail = (f", and both beat the best simple rule ({f(best_simple)}), so the simpler model is enough" if min(rfm["auc"], lrm["auc"]) - best_simple > 0.02
            else f", and neither clearly beats the best simple rule ({f(best_simple)})")
    head = f"Random Forest (AUC {f(rfm['auc'])}) and Logistic Regression (AUC {f(lrm['auc'])}) performed about the same{tail}."
else:
    a, b = (("Random Forest", rfm), ("Logistic Regression", lrm)) if rfm["auc"] > lrm["auc"] else (("Logistic Regression", lrm), ("Random Forest", rfm))
    head = f"{a[0]} reached AUC {f(a[1]['auc'])}, against {f(b[1]['auc'])} for {b[0]} and {f(best_simple)} for the best simple rule."
rows = "".join(f"<tr><td>{n}</td><td>{f(r['auc'])}</td><td>{f(r['p50'], 2)}</td><td>{f(r['p500'], 2)}</td><td>{f(cv[n][0])} ± {f(cv[n][1])}</td></tr>" for n, r in res.items())
trows = "".join(f"<tr><td>{i+1}</td><td>{r.action}</td><td><code>{r.reason_code}</code></td><td>{int(r.imp_march):,}</td><td>{f(r.ctr, 4)}</td><td>{f(r.pos_march, 1)}</td><td>{f(r.oof_score, 2)}</td></tr>" for i, r in top10.reset_index(drop=True).iterrows())
topf = ", ".join(f"<code>{k}</code> ({f(v)})" for k, v in imp.head(3).items())
V = dict(n_items=f"{stats['n_items']:,}", n_clients=stats["n_clients"], n_daily=f"{stats['n_daily']:,}", ga4=f(stats["ga4_pct"], 1),
         base=f(stats["base_rate"]), tbase=f(split["test_base"]), tclients=split["test_clients"], trows=f"{split['test_rows']:,}",
         head=head, rows=rows, trows_html=trows, topf=topf, auc_r=f(audit["random"]), auc_g=f(audit["grouped"]), auc_l=f(audit["leak"]),
         unk=f(lim["unk_stale"], 1), zc=f(lim["zero_ctr"], 1), md=lim["modal_days"], mp=f(lim["modal_pct"], 1), thr=f(thr, 2),
         c1=counts.get("stale_high_risk", 0), c2=counts.get("low_ctr_high_risk", 0), c3=counts.get("high_risk_unclear_cause", 0),
         p500=f(value["p500"], 2), tz=f(value["top_zero"], 0), pl1=f(lim["pos_lt1"], 1), gap=f(abs(audit["random"] - audit["grouped"])), auc_rf=f(rfm["auc"]), p500rf=f(rfm["p500"], 2), extra=f"{value['extra']:.0f}", wasted=f"{value['wasted']:.0f}",
         img1=b64("work/figures/capstone_model_vs_baseline.png"), img2=b64("work/figures/capstone_leakage_audit.png"))

PAGE = """<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Which pages should you refresh first? Ranking next-month search decline from one month of Search Console data</title>
<style>
:root{--bg:#f7f8fa;--ink:#14181f;--mut:#5a6472;--acc:#1d4ed8;--rule:#d9dde3;--soft:#eaeef5}
@media(prefers-color-scheme:dark){:root{--bg:#0f1318;--ink:#e8ebf0;--mut:#9aa4b2;--acc:#7da2ff;--rule:#2a323d;--soft:#1a212b}}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:18px/1.65 Charter,"Iowan Old Style",Georgia,serif}
main{max-width:46rem;margin:0 auto;padding:2.5rem 1.2rem 5rem}
h1,h2,h3,nav,table,figcaption,.meta{font-family:system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
h1{font-size:2rem;line-height:1.2;margin:0 0 .5rem}h2{font-size:1.35rem;margin:2.6rem 0 .6rem;padding-top:1rem;border-top:1px solid var(--rule)}h3{font-size:1.05rem;margin:1.6rem 0 .3rem}
.meta{color:var(--mut);font-size:.9rem}.abs{background:var(--soft);border-left:4px solid var(--acc);padding:1rem 1.2rem;margin:1.5rem 0}
nav{font-size:.9rem;margin:1rem 0;display:flex;flex-wrap:wrap;gap:.4rem 1rem}a{color:var(--acc)}
table{border-collapse:collapse;width:100%;font-size:.88rem;display:block;overflow-x:auto}th,td{padding:.4rem .6rem;border-bottom:1px solid var(--rule);text-align:left;white-space:nowrap}
tr.best td{font-weight:700;background:var(--soft)}img{max-width:100%;height:auto}figure{margin:1.5rem 0}figcaption{font-size:.88rem;color:var(--mut)}
code{font-size:.85em;background:var(--soft);padding:.05em .3em}a:focus-visible{outline:3px solid var(--acc)}
</style></head><body><main>
<h1>Which pages should you refresh first? Ranking next-month search decline from one month of Search Console data</h1>
<p class="meta">Capstone research paper · FlyRank ML Internship track · Lane: Content Refresh Prioritization</p>
<nav><a href="#intro">Introduction</a><a href="#data">Data</a><a href="#method">Methodology</a><a href="#results">Results</a><a href="#limits">Limitations</a><a href="#recs">Recommendations</a><a href="#repro">Reproducibility</a><a href="#credit">Credit</a></nav>
<div class="abs"><strong>Abstract.</strong> Content teams cannot refresh every page, so which ones should they look at first? I used one month (March 2026) of daily Search Console metrics for @@n_items@@ content items across @@n_clients@@ client sites to rank pages by the chance their impressions fall by 20% or more the next month. I compared a transparent rule, a one-feature rule, Logistic Regression and Random Forest on a split where each client appears only in train or only in test. @@head@@ The ranking is a decision-support aid for human reviewers, not a forecast and not evidence that a refresh will recover traffic.</div>

<h2 id="intro">Introduction</h2>
<p>An editor with time for 500 pages a week needs an order to review them in. Picture a site with thousands of pages. In this data, @@base@@ of pages lost at least 20% of their impressions in a month, so they cannot all be treated as urgent, and reading every one is not possible.</p>
<p>This paper asks whether signals visible in one month of search data can put the pages that go on to lose visibility nearer the top of that list, and whether a model does so better than a simple rule.</p>

<h2 id="data">Data</h2>
<p><strong>Source.</strong> The FlyRank ML Internship warehouse release (access-gated). <strong>Tables used.</strong> <code>fact_content_daily_performance</code> (daily search impressions, clicks and average position per content item and client; @@n_daily@@ daily rows across the two months used) and <code>dim_content</code> (content update date).</p>
<p><strong>Windows.</strong> Features come from March 2026 only; the outcome is April 2026. The final month of the release was kept sealed and never used.</p>
<p><strong>Kept.</strong> Content items with at least 100 March impressions and a measured position: @@n_items@@ items, @@n_clients@@ clients. <strong>Excluded and why.</strong> Product-computed flags and scores (they would be circular); identifiers as model inputs (used only to group clients); Google Analytics fields, available for only @@ga4@@% of March rows.</p>

<h2 id="method">Methodology</h2>
<p><strong>Label.</strong> An item “declined” if its April impressions were below 80% of its March impressions (base rate @@base@@). This is an observed proxy for decline, not a cause.</p>
<p><strong>Features (March only).</strong> Impressions, click-through rate, average position, days with impressions, days since last content update, and a flag for unknown update date. Update dates falling after March were treated as unknown, because they cannot have been known in March.</p>
<p><strong>Baselines.</strong> The week-4 rule scores pages that are at least 180 days stale and have 500+ impressions. A second rule ranks lowest click-through rate first; it is the bar any model should clear.</p>
<p><strong>Validation.</strong> The primary test set holds out whole clients (@@tclients@@ clients, @@trows@@ items; test base rate @@tbase@@). Five client-grouped folds check stability. Metrics: AUC, and precision among the top 50 and top 500 ranked items, shown next to the base rate.</p>
<p><strong>Leakage checks.</strong> No April-derived column is a feature; no product flags; identifiers excluded. Random Forest AUC was @@auc_r@@ under a random split and @@auc_g@@ under the client-grouped split. As a harness check, deliberately adding April impressions raised AUC to @@auc_l@@, so a real leak would have been visible.</p>
<figure><img alt="Bar chart of Random Forest AUC under a random split, a grouped split and a deliberate leak" src="data:image/png;base64,@@img2@@"><figcaption>A deliberate leak lifts AUC sharply, confirming the checks can see leakage. Gap between the random and grouped designs: @@gap@@ AUC.</figcaption></figure>

<h2 id="results">Results</h2>
<table><thead><tr><th>Method</th><th>AUC</th><th>Precision@50</th><th>Precision@500</th><th>5-fold AUC (mean ± sd)</th></tr></thead><tbody>@@rows@@</tbody></table>
<p class="meta">Same grouped test split for every row; test base rate @@tbase@@. Top Random Forest features by importance: @@topf@@.</p>
<figure><img alt="Bars comparing AUC and precision at 500 for four methods" src="data:image/png;base64,@@img1@@"><figcaption>@@head@@ Compare each bar with the dashed reference: chance for AUC, the base rate for precision.</figcaption></figure>
<p>The week-4 rule scores most pages zero, so its AUC sits near chance by construction; precision is the fairer read of it. Gaps smaller than the fold-to-fold spread (the ± column) should not be read as real differences. With only @@tclients@@ held-out clients on the primary split, treat every gap as directional.</p>

<h2 id="limits">Limitations and honest framing</h2>
<p><strong>One month-pair.</strong> Results come from March to April 2026 and @@n_clients@@ clients. Seasonality, algorithm changes or demand shifts could cause declines the features cannot see.</p>
<p><strong>A proxy label.</strong> A 20% impression drop is a convention, not a verdict on content quality. No causal claim is made: nothing here shows a refresh would reverse a decline.</p>
<p><strong>Weak staleness data.</strong> @@unk@@% of items had no update date verifiable as of March, so staleness-based reasons rarely fire. Among known dates, @@mp@@% share the same value (@@md@@ days), which looks like a pipeline artifact rather than real editing history.</p>
<p><strong>Possible measurement artifacts.</strong> @@zc@@% of items show exactly zero clicks despite 100+ impressions, and @@tz@@% of the top 500 ranked items are among them, so the highest-ranked pages may reflect missing click data rather than real behavior. @@pl1@@% of items also have an average position below 1, which cannot occur in raw Search Console data, so position is best read as a relative signal. Check measurement before acting.</p>
<p><strong>Limited signal.</strong> Random Forest AUC was @@auc_rf@@ and precision@500 was @@p500rf@@ against a @@tbase@@ base rate, so many top-ranked pages still did not decline. Use it as a ranking aid with human review, a directional starting point only.</p>

<h2 id="recs">Ranked recommendations</h2>
<p>Every item is scored out-of-fold (by a model that never saw its client). The top 10% by score (score at least @@thr@@) are “high risk” and get an action by the strongest visible reason.</p>
<table><thead><tr><th>Reason code</th><th>Action</th><th>Items</th></tr></thead><tbody>
<tr><td><code>stale_high_risk</code></td><td>Review for content refresh</td><td>@@c1@@</td></tr>
<tr><td><code>low_ctr_high_risk</code></td><td>Review title/snippet and CTA</td><td>@@c2@@</td></tr>
<tr><td><code>high_risk_unclear_cause</code></td><td>Manual review (cause unclear)</td><td>@@c3@@</td></tr></tbody></table>
<h3>Top 10 of the ranked queue (anonymized)</h3>
<table><thead><tr><th>Rank</th><th>Action</th><th>Reason</th><th>Impressions</th><th>CTR</th><th>Position</th><th>Score</th></tr></thead><tbody>@@trows_html@@</tbody></table>
<h3>Cost and value</h3>
<p>Reviewing the top 500 items surfaced pages that declined @@p500@@ of the time versus a @@base@@ base rate: about @@extra@@ more declining pages found than random review, and about @@wasted@@ of 500 reviews on pages that did not decline. This measures ranking value, not traffic recovered.</p>
<h3>Human review required</h3>
<p>Before acting, check whether the page is deliberately frozen or evergreen, was recently optimized, or has a business reason (season, redesign, compliance) for the pattern, or may have missing click tracking.</p>
<h3>Do not automate</h3>
<p>No auto-publishing, unpublishing or deindexing from this score; no sole use for resourcing decisions; no use on other clients or lanes without re-validation; the score is a ranking signal, not a certified probability.</p>
<h3>Monitoring</h3>
<p>Re-validate monthly on the newest month-pair. Retrain if AUC falls below 0.60, if the monthly decline base rate moves by more than 10 points, or if unknown update dates keep growing.</p>

<h2 id="repro">Reproducibility</h2>
<p>All code is in the public repository under <code>work/notebooks/</code>; the final notebook is <code>capstone.ipynb</code>. Seed 42 throughout; queries are ordered so reruns match. Access to the warehouse release is required to rerun.</p>
<p><a href="https://github.com/binary-cpu/flyrank-ml-track-david">Repository: binary-cpu/flyrank-ml-track-david</a></p>

<h2 id="credit">Acknowledgments and data credit</h2>
<p>Built on the <a href="https://flyrank.ai">FlyRank ML Internship dataset</a>. Thanks to the FlyRank team for the data and the track.</p>
</main></body></html>"""
for k, v in V.items(): PAGE = PAGE.replace(f"@@{k}@@", str(v))
assert "@@" not in PAGE, "unfilled placeholder in page"
open("docs/index.html", "w").write(PAGE)

metrics = dict(stats=stats, split=split, results=res, cv_auc={k: list(v) for k, v in cv.items()}, audit=audit, limits=lim,
               tiers=counts, value=value, importance=imp.round(4).to_dict(), feature_month=FEATURE_MONTH, outcome_month=OUTCOME_MONTH)
json.dump(metrics, open("work/outputs/capstone_metrics.json", "w"), indent=2, default=float)
queue.drop(columns=["content_hash_id"]).to_csv("work/outputs/capstone_queue.csv", index=False)   # stays out of git
with zipfile.ZipFile("paper_bundle.zip", "w") as z:
    for p in ["docs/index.html", "work/figures/capstone_model_vs_baseline.png", "work/figures/capstone_leakage_audit.png", "work/outputs/capstone_metrics.json"]:
        z.write(p)
try:
    from google.colab import files; files.download("paper_bundle.zip")
except Exception:
    print("Not in Colab: find paper_bundle.zip next to the notebook.")
print(f"page: {len(PAGE)/1024:.0f} KB")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

page: 113 KB


In [8]:
pub = open("docs/index.html").read()
needed = ["Abstract", "Introduction", "Data", "Methodology", "Results", "Limitations", "Ranked recommendations", "Reproducibility", "Acknowledgments", "https://flyrank.ai"]
print({k: (k in pub) for k in needed})
import re
print("token-like strings in page:", bool(re.search(r"hf_[A-Za-z0-9]{20,}", pub)), "| content/client ids in page:", bool(re.search(r"(content|client)_[0-9a-f]{12,}", pub)))

{'Abstract': True, 'Introduction': True, 'Data': True, 'Methodology': True, 'Results': True, 'Limitations': True, 'Ranked recommendations': True, 'Reproducibility': True, 'Acknowledgments': True, 'https://flyrank.ai': True}
token-like strings in page: False | content/client ids in page: False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.